In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

from scipy.stats import skew, mode
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import pydicom

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

tf = None

SEED = 1337


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    if tf is not None:
        tf.random.set_seed(seed)




In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f"Training Set Memory Usage = {df_train.memory_usage().sum() / 1024 ** 2:.2f} MB")
print(f'Set Test Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}')
print(f"Test Set Memory Usage = {df_test.memory_usage().sum() / 1024 ** 2:.2f} MB")
print(f"Sample Submission Shape = {df_submission.shape}")
print(
    f"Sample Submission Memory Usage = {df_submission.memory_usage().sum() / 1024 ** 2:.2f} MB"
)




Training Set Shape = (1394, 7) - Patients = 158
Training Set Memory Usage = 0.07 MB
Set Test Shape = (18, 7) - Patients = 18
Test Set Memory Usage = 0.00 MB
Sample Submission Shape = (1908, 3)
Sample Submission Memory Usage = 0.04 MB


In [3]:
class Preprocessor:

    def __init__(
        self, df_train, df_test, df_submission, n_folds, shuffle, resize_shape
    ):
        self.df_train = df_train.copy(deep=True)
        self.df_train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.df_test = df_test.copy(deep=True)
        self.df_submission = df_submission.copy(deep=True)
        self.n_folds = n_folds
        self.shuffle = shuffle
        self.resize_shape = resize_shape

    def _drop_duplicates(self):
        self.df_train["FVC"] = self.df_train.groupby(["Patient", "Weeks"])[
            "FVC"
        ].transform("mean")
        self.df_train["Percent"] = self.df_train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.df_train.drop_duplicates(inplace=True)
        self.df_train.reset_index(drop=True, inplace=True)

    def _label_encode(self):
        for df in [self.df_train, self.df_test]:
            df["Sex"] = df["Sex"].map({"Male": 0, "Female": 1})
            df["SmokingStatus"] = df["SmokingStatus"].map(
                {"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2}
            )

    def _create_folds(self):
        self.df_train["Sex_SmokingStatus"] = (
            self.df_train["Sex"].astype(str)
            + "_"
            + self.df_train["SmokingStatus"].astype(str)
        )
        for group in self.df_train["Sex_SmokingStatus"].unique():
            patients = self.df_train[self.df_train["Sex_SmokingStatus"] == group][
                "Patient"
            ].unique()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)
            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV1_Fold"
                ] = fold

        for patient_name in self.df_train["Patient"].unique():
            fvc_vals = self.df_train[self.df_train["Patient"] == patient_name][
                "FVC"
            ].values[-2:]
            if fvc_vals.std() == 0:
                z = np.zeros_like(fvc_vals)
            else:
                z = (fvc_vals - fvc_vals.mean()) / fvc_vals.std()
            reg = LinearRegression().fit(
                self.df_train[self.df_train["Patient"] == patient_name]["Weeks"]
                .values[-2:]
                .reshape(-1, 1),
                z,
            )
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Intercept"] = (
                reg.intercept_
            )
            self.df_train.loc[self.df_train["Patient"] == patient_name, "Coef"] = (
                reg.coef_[0]
            )

        self.df_train.loc[self.df_train["Coef"] > 0.4, "Cluster"] = 1
        self.df_train.loc[
            (self.df_train["Coef"] < 0.4) & (self.df_train["Coef"] > -0.4), "Cluster"
        ] = 2
        self.df_train.loc[self.df_train["Coef"] < -0.4, "Cluster"] = 3

        for group in self.df_train["Cluster"].unique():
            patients = self.df_train[self.df_train["Cluster"] == group][
                "Patient"
            ].unique()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)
            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.df_train.loc[
                    self.df_train["Patient"].isin(patient_group), "CV2_Fold"
                ] = fold

        patients = self.df_train["Patient"].unique()
        np.random.seed(SEED)
        np.random.shuffle(patients)
        for fold, patient_group in enumerate(np.array_split(patients, self.n_folds), 1):
            self.df_train.loc[
                self.df_train["Patient"].isin(patient_group), "CV3_Fold"
            ] = fold

        self.df_train.drop(
            columns=["Sex_SmokingStatus", "Intercept", "Coef", "Cluster"], inplace=True
        )

    def load_scan(self, dataset, patient_name):
        patient_dir = (
            f"../input/osic-pulmonary-fibrosis-progression/{dataset}/{patient_name}"
        )
        slices = [
            pydicom.dcmread(os.path.join(patient_dir, s))
            for s in os.listdir(patient_dir)
        ]
        try:
            slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))
            slice_positions = np.round([s.ImagePositionPatient[2] for s in slices], 4)
            non_dup = np.unique(
                [np.where(sp == slice_positions)[0][0] for sp in slice_positions]
            )
        except AttributeError:
            slices.sort(key=lambda s: int(s.InstanceNumber))
            instance_numbers = np.array([int(s.InstanceNumber) for s in slices])
            non_dup = np.unique(
                [np.where(inum == instance_numbers)[0][0] for inum in instance_numbers]
            )
        slices = list(np.array(slices)[non_dup])
        metadata = {}
        pixel_spacings = np.zeros((len(slices), 2))
        slice_positions = np.zeros((len(slices)))
        for i, s in enumerate(slices):
            try:
                pixel_spacings[i, :] = np.array(s.PixelSpacing)
            except AttributeError:
                pixel_spacings[i, :] = np.nan
            try:
                slice_positions[i] = s.ImagePositionPatient[2]
            except AttributeError:
                pass
        metadata["PixelSpacing"] = list(np.round(pixel_spacings.mean(axis=0), 3))
        if patient_name == "ID00128637202219474716089":
            metadata["SliceSpacing"] = 5.0
        elif patient_name == "ID00132637202222178761324":
            metadata["SliceSpacing"] = 0.7
        else:
            metadata["SliceSpacing"] = list(
                mode(np.abs(np.diff(np.round(slice_positions, 3))))
            )[0][0]
        scan = np.zeros(
            (len(slices), self.resize_shape[0], self.resize_shape[1]), dtype=np.int16
        )
        for i, s in enumerate(slices):
            cropped = self.crop_slice(s.pixel_array)
            resized = self.resize_slice(cropped)
            if not np.all(resized == 0):
                scan[i] = np.int16(resized)
        del slices
        scan = scan[~np.all(scan == 0, axis=(-1, -2))]
        return scan, metadata

    def crop_slice(self, s):
        if np.all(s == 0):
            return s
        if s.shape[0] != self.resize_shape[0] and s.shape[1] != self.resize_shape[1]:
            s = s[~np.all(s == 0, axis=1)]
            s = s[:, ~np.all(s == 0, axis=0)]
        return s

    def resize_slice(self, s):
        if s.shape[0] != self.resize_shape[0] and s.shape[1] != self.resize_shape[1]:
            return cv2.resize(s, self.resize_shape, interpolation=cv2.INTER_NEAREST)
        return s

    def _create_baseline_features(self):
        self.df_submission["Type"] = "Test"
        self.df_submission["Patient"] = (
            self.df_submission["Patient_Week"]
            .apply(lambda x: x.split("_")[0])
            .astype(str)
        )
        self.df_submission["Weeks"] = self.df_submission["Patient_Week"].apply(
            lambda x: int(x.split("_")[1])
        )
        self.df_submission.drop(
            columns=["Patient_Week", "FVC", "Confidence"], inplace=True
        )

        self.df_train["Type"] = "Train"
        self.df_train["Weeks_Passed"] = self.df_train["Weeks"] - self.df_train.groupby(
            "Patient"
        )["Weeks"].transform("min")
        self.df_train["FVC_Baseline"] = self.df_train.groupby("Patient")[
            "FVC"
        ].transform("first")

        for patient in self.df_test["Patient"].unique():
            fvc_val = self.df_test.loc[
                self.df_test["Patient"] == patient, "FVC"
            ].values[0]
            perc_val = self.df_test.loc[
                self.df_test["Patient"] == patient, "Percent"
            ].values[0]
            age_val = self.df_test.loc[
                self.df_test["Patient"] == patient, "Age"
            ].values[0]
            sex_val = self.df_test.loc[
                self.df_test["Patient"] == patient, "Sex"
            ].values[0]
            smoke_val = self.df_test.loc[
                self.df_test["Patient"] == patient, "SmokingStatus"
            ].values[0]

            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "FVC_Baseline"
            ] = fvc_val
            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "Percent"
            ] = perc_val
            self.df_submission.loc[self.df_submission["Patient"] == patient, "Age"] = (
                age_val
            )
            self.df_submission.loc[self.df_submission["Patient"] == patient, "Sex"] = (
                sex_val
            )
            self.df_submission.loc[
                self.df_submission["Patient"] == patient, "SmokingStatus"
            ] = smoke_val

        self.df_submission["Weeks_Passed"] = self.df_submission["Weeks"]

        self.df_all = pd.concat(
            [self.df_train, self.df_submission], ignore_index=True, axis=0
        )
        self.df_all["Age"] += np.int8(np.floor(self.df_all["Weeks_Passed"] / 52))

        self.df_all["Weeks"] = self.df_all["Weeks"].astype(np.int16)
        self.df_all["Age"] = self.df_all["Age"].astype(np.float32)
        self.df_all["FVC_Baseline"] = self.df_all["FVC_Baseline"].astype(np.float32)
        self.df_all["Percent"] = self.df_all["Percent"].astype(np.float32)
        self.df_all["Weeks_Passed"] = self.df_all["Weeks_Passed"].astype(np.float32)
        self.df_all["Sex"] = self.df_all["Sex"].astype(np.uint8)
        self.df_all["SmokingStatus"] = self.df_all["SmokingStatus"].astype(np.uint8)
        self.df_all["FVC"] = self.df_all["FVC"].astype(np.float32)

        self.df_train = self.df_all.loc[self.df_all["Type"] == "Train", :].drop(
            columns=["Type"]
        )
        for i in range(1, 4):
            self.df_train[f"CV{i}_Fold"] = self.df_train[f"CV{i}_Fold"].astype(np.uint8)
        self.df_test = self.df_all.loc[self.df_all["Type"] == "Test", :].drop(
            columns=["Type", "FVC", "CV1_Fold", "CV2_Fold", "CV3_Fold"]
        )

    def get_data(self):
        self._drop_duplicates()
        self._label_encode()
        self._create_folds()
        self._create_baseline_features()
        print(f"Preprocessed Training Set Shape = {self.df_train.shape}")
        print(
            f"Preprocessed Training Set Memory Usage = {self.df_train.memory_usage().sum() / 1024 ** 2:.2f} MB"
        )
        print(f"Preprocessed Test Set Shape = {self.df_test.shape}")
        print(
            f"Preprocessed Test Set Memory Usage = {self.df_test.memory_usage().sum() / 1024 ** 2:.2f} MB"
        )
        return self.df_train.copy(deep=True), self.df_test.copy(deep=True)




In [4]:
preprocessor_parameters = {
    "df_train": df_train,
    "df_test": df_test,
    "df_submission": df_submission,
    "n_folds": 2,
    "shuffle": True,
    "resize_shape": (512, 512),
}
preprocessor = Preprocessor(**preprocessor_parameters)
df_train, df_test = preprocessor.get_data()




Preprocessed Training Set Shape = (1387, 12)
Preprocessed Training Set Memory Usage = 0.06 MB
Preprocessed Test Set Shape = (1908, 8)
Preprocessed Test Set Memory Usage = 0.07 MB


In [5]:
seed_everything(SEED)

df_train["Weeks_Age"] = df_train["Weeks"] * df_train["Age"]
df_test["Weeks_Age"] = df_test["Weeks"] * df_test["Age"]

feature_cols = [
    "Weeks",
    "Age",
    "Sex",
    "SmokingStatus",
    "Percent",
    "FVC_Baseline",
    "Weeks_Age",
    "Weeks_Passed",
]

X_train = df_train[feature_cols]
y_train = df_train["FVC"]

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

test_preds = np.zeros(len(df_test), dtype=np.float32)
oof_preds = np.zeros(len(df_train), dtype=np.float32)

for train_idx, val_idx in kf.split(X_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    gbr_fold = GradientBoostingRegressor(
        n_estimators=2500,
        learning_rate=0.04,
        max_depth=7,
        subsample=0.9,
        random_state=SEED,
    )
    gbr_fold.fit(X_tr, y_tr)

    test_preds += gbr_fold.predict(df_test[feature_cols])
    oof_preds[val_idx] = gbr_fold.predict(X_val)

df_test["FVC"] = test_preds / kf.n_splits

gbr_full = GradientBoostingRegressor(
    n_estimators=2500,
    learning_rate=0.04,
    max_depth=7,
    subsample=0.9,
    random_state=SEED,
)
gbr_full.fit(X_train, y_train)
df_test["FVC"] = gbr_full.predict(df_test[feature_cols])

# Compute residual standard deviation from out‑of‑fold predictions
residual_std = np.std(y_train - oof_preds)

# Increase confidence multiplier to raise σ (helps the Laplace metric)
# Ensure a minimum of 70 and also enforce a floor of 200 to guarantee a larger σ.
conf_val = max(70.0, residual_std * 4.0, 200.0)
df_test["Confidence"] = conf_val

df_test["Patient_Week"] = (
    df_test["Patient"].astype(str) + "_" + df_test["Weeks"].astype(str)
)




In [6]:
class SubmissionPipeline:

    def __init__(self, df_train, df_test):
        self.df_train = df_train
        self.df_test = df_test

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma = np.maximum(sigma, 70)
        delta = np.minimum(np.abs(y_true - y_pred), 1000)
        return np.mean(-np.sqrt(2) * delta / sigma - np.log(np.sqrt(2) * sigma))

    def single_model(self, model_prefix):
        pred_cols = [c for c in self.df_train.columns if c.startswith(model_prefix)]
        if pred_cols:
            if model_prefix.split("_")[1] == "MLP":
                fvc_col, conf_col = sorted(pred_cols)
                score = self.laplace_log_likelihood_metric(
                    self.df_train["FVC"],
                    self.df_train[fvc_col],
                    self.df_train[conf_col],
                )
                print(f"Single Model {model_prefix} Score: {score:.6}")
                self.df_test["FVC"] = self.df_test[fvc_col]
                self.df_test["Confidence"] = self.df_test[conf_col]
            else:  # QR
                quantiles = (
                    self.qr_parameters["quantiles"]
                    if hasattr(self, "qr_parameters")
                    else [0.25, 0.5, 0.75]
                )
                lower_q, median_q, upper_q = sorted(quantiles)
                lower_col = f"{model_prefix}_{lower_q}_Predictions"
                median_col = f"{model_prefix}_{median_q}_Predictions"
                upper_col = f"{model_prefix}_{upper_q}_Predictions"
                if not all(
                    col in self.df_train.columns
                    for col in [lower_col, median_col, upper_col]
                ):
                    cols_sorted = sorted(pred_cols)
                    lower_col, median_col, upper_col = cols_sorted[:3]
                score = self.laplace_log_likelihood_metric(
                    self.df_train["FVC"],
                    self.df_train[median_col],
                    self.df_train[upper_col] - self.df_train[lower_col],
                )
                print(f"Single Model {model_prefix} Score: {score:.6}")
                self.df_test["FVC"] = self.df_test[median_col]
                self.df_test["Confidence"] = (
                    self.df_test[upper_col] - self.df_test[lower_col]
                )
        else:
            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"], self.df_test["FVC"], self.df_test["Confidence"]
            )
            print(f"Baseline Model Score (no prefix columns): {score:.6}")

        print(self.df_test[["Patient_Week", "FVC", "Confidence"]].describe())
        return self.df_test[["Patient_Week", "FVC", "Confidence"]].copy(deep=True)




In [7]:
sub = SubmissionPipeline(df_train, df_test)
df_submission = sub.single_model("CV1_QR")
df_submission.to_csv("submission.csv", index=False)

Baseline Model Score (no prefix columns): nan
               FVC   Confidence
count  1908.000000  1908.000000
mean   2949.014934   418.402252
std     883.614873     0.000000
min    1546.708112   418.402252
25%    2157.873350   418.402252
50%    3082.389978   418.402252
75%    3341.128353   418.402252
max    5033.467755   418.402252
